# Intelligent Agents #

This notebook serves as supporting material for topics covered in **Chapter 2 - Intelligent Agents** from the book *Artificial Intelligence: A Modern Approach.* This notebook uses implementations from [agents.py](https://github.com/aimacode/aima-python/blob/master/agents.py) module. Let's start by importing everything from agents module.

In [27]:
from agents import *
from notebook import psource

## CONTENTS

* Overview
* Agent
* Environment
* Simple Agent and Environment
* Agents in a 2-D Environment
* Wumpus Environment

## OVERVIEW

An agent, as defined in 2.1, is anything that can perceive its <b>environment</b> through sensors, and act upon that environment through actuators based on its <b>agent program</b>. This can be a dog, a robot, or even you. As long as you can perceive the environment and act on it, you are an agent. This notebook will explain how to implement a simple agent, create an environment, and implement a program that helps the agent act on the environment based on its percepts.

## AGENT

Let us now see how we define an agent. Run the next cell to see how `Agent` is defined in agents module.

In [28]:
psource(Agent)

The `Agent` has two methods.
* `__init__(self, program=None)`: The constructor defines various attributes of the Agent. These include

    * `alive`: which keeps track of whether the agent is alive or not 
    
    * `bump`: which tracks if the agent collides with an edge of the environment (for eg, a wall in a park)
    
    * `holding`: which is a list containing the `Things` an agent is holding, 
    
    * `performance`: which evaluates the performance metrics of the agent 
    
    * `program`: which is the agent program and maps an agent's percepts to actions in the environment. If no implementation is provided, it defaults to asking the user to provide actions for each percept.
    
* `can_grab(self, thing)`: Is used when an environment contains things that an agent can grab and carry. By default, an agent can carry nothing.

## ENVIRONMENT
Now, let us see how environments are defined. Running the next cell will display an implementation of the abstract `Environment` class.

In [29]:
psource(Environment)

`Environment` class has lot of methods! But most of them are incredibly simple, so let's see the ones we'll be using in this notebook.

* `thing_classes(self)`: Returns a static array of `Thing` sub-classes that determine what things are allowed in the environment and what aren't

* `add_thing(self, thing, location=None)`: Adds a thing to the environment at location

* `run(self, steps)`: Runs an environment with the agent in it for a given number of steps.

* `is_done(self)`: Returns true if the objective of the agent and the environment has been completed

The next two functions must be implemented by each subclasses of `Environment` for the agent to recieve percepts and execute actions 

* `percept(self, agent)`: Given an agent, this method returns a list of percepts that the agent sees at the current time

* `execute_action(self, agent, action)`: The environment reacts to an action performed by a given agent. The changes may result in agent experiencing new percepts or other elements reacting to agent input.

หัวใจหลักของการสร้าง **Agent** และ **Environment** ในไลบรารีนี้คือการทำความเข้าใจคลาส (Class) พื้นฐาน 3 อย่าง และการทำงานร่วมกันของมัน ได้แก่ **Thing, Agent, และ Environment**

องค์ประกอบหลักจาก agents.py
ทุกอย่างในโลกจำลองของเราสร้างขึ้นจากคลาสพื้นฐานเหล่านี้:


**Thing 🧸**
นี่คือคลาสแม่บทสำหรับวัตถุทุกชนิดที่สามารถมีอยู่ใน Environment ได้. ไม่ว่าจะเป็นกำแพง, อาหาร, หรือแม้กระทั่งตัว Agent เอง ก็ล้วนสืบทอดคุณสมบัติมาจาก Thing ทั้งสิ้น.


**Agent 🤖**
Agent เป็น Thing ชนิดพิเศษที่มีความสามารถในการรับรู้ (perceive) และกระทำ (act). สิ่งที่ทำให้ Agent แตกต่างคือแอตทริบิวต์ program ซึ่งเป็นฟังก์ชันที่รับข้อมูลการรับรู้ (percept) จาก Environment เข้ามา แล้วส่งคืนเป็นการกระทำ (action) กลับไป. นอกจากนี้ Agent ยังมีสถานะต่างๆ เช่น alive (ยังมีชีวิตอยู่หรือไม่) และ performance (คะแนนประสิทธิภาพ).


**Environment 🏞️**
นี่คือโลกหรือฉากที่ Agent อาศัยอยู่. Environment มีหน้าที่สำคัญสองอย่างคือ:

percept(self, agent): สร้างและส่งข้อมูลการรับรู้ (percepts) ที่เหมาะสมให้กับ Agent ณ เวลานั้นๆ.

execute_action(self, agent, action): รับการกระทำ (action) จาก Agent มา แล้วปรับเปลี่ยนสถานะของสิ่งแวดล้อมให้สอดคล้องกับการกระทำนั้น.

**วงจรการทำงานพื้นฐาน (The Simulation Loop)**
การจำลอง (simulation) จะดำเนินไปเป็นวงจรในแต่ละขั้นเวลา (time step) ซึ่งขับเคลื่อนโดยเมธอด run() ของ Environment. ในแต่ละขั้นจะเกิดเหตุการณ์ตามลำดับดังนี้:

* รับรู้ (Perceive): Environment จะเรียกใช้เมธอด percept() เพื่อตรวจสอบว่า Agent รับรู้อะไรได้บ้าง เช่น "เห็นอาหารอยู่ตรงหน้า" หรือ "ชนกำแพง"

* คิด (Think): ข้อมูลการรับรู้ (percept) ที่ได้จากขั้นตอนที่ 1 จะถูกส่งเข้าไปยัง program ของ Agent

* ตัดสินใจ (Decide): program ของ Agent จะประมวลผลข้อมูลการรับรู้และตัดสินใจว่าจะทำอะไรต่อ แล้วส่งคืนเป็นการกระทำ (action) เช่น 'eat' หรือ 'move forward'

* กระทำ (Act): Environment จะนำ action ที่ได้จากขั้นตอนที่ 3 ไปประมวลผลในเมธอด execute_action() ซึ่งจะทำให้โลกเกิดการเปลี่ยนแปลง เช่น อาหารถูกลบไป หรือตำแหน่งของ Agent เปลี่ยนไป

* วงจรนี้จะวนซ้ำไปเรื่อยๆ จนกว่าจะถึงเงื่อนไขสิ้นสุดที่กำหนดไว้ในเมธอด is_done().

**การเชื่อมโยงไปยังตัวอย่างใน agents.ipynb**
เมื่อเข้าใจหลักการพื้นฐานนี้แล้ว คุณจะเห็นว่าตัวอย่าง BlindDog ในไฟล์ agents.ipynb ก็ใช้โครงสร้างเดียวกันนี้:

* Agent: BlindDog คือ Agent ของเรา.

* Environment: Park คือโลกที่สุนัขอาศัยอยู่.

* Percepts: เมธอด percept() ของ Park จะคืนค่าเป็นรายการของ Thing ที่อยู่ในตำแหน่งเดียวกับสุนัข.

* Program: ฟังก์ชัน program() ที่เราสร้างขึ้นจะรับรายการของ Thing เข้ามา แล้วตัดสินใจว่าจะ 'eat' (ถ้าเจอ Food), 'drink' (ถ้าเจอ Water), หรือ 'move down' (ถ้าไม่เจออะไร).

* Actions: เมธอด execute_action() ของ Park จะจัดการกับการกระทำเหล่านี้ เช่น การลบอาหาร ( delete_thing() ) หรือการเปลี่ยนตำแหน่งของสุนัข.

**ในตัวอย่างนี้ เราจะสร้างโลกง่ายๆ แบบ 1 มิติ ที่มี "นักสะสม" (Agent) ซึ่งมีเป้าหมายเดียวคือเดินไปเก็บ "สมบัติ" (Thing)**


**วิธีทำความเข้าใจโค้ดนี้**
* ส่วนที่ 1: เป็นการจำลองคลาสพื้นฐานจาก agents.py เพื่อให้โค้ดนี้สามารถทำงานได้ด้วยตัวเองโดยไม่ต้อง import ไฟล์อื่น

* ส่วนที่ 2: เราได้สร้างคลาส Treasure ที่สืบทอดจาก Thing และ LineWorld ที่สืบทอดจาก Environment ซึ่งเป็นโลกจำลองของเรา

* ส่วนที่ 3: collector_program คือ "สมอง" ของ Agent ที่เราเขียนขึ้นมา เป็นฟังก์ชันง่ายๆ ที่รับ percepts และคืนค่า action

* ส่วนที่ 4: เป็นส่วนที่นำทุกอย่างมารวมกัน สร้างอ็อบเจกต์ต่างๆ, กำหนดตำแหน่งเริ่มต้น, และสั่งให้ world.run() เพื่อเริ่มการจำลอง


เมื่อคุณรันโค้ดนี้ จะเห็น Log การทำงานในแต่ละขั้นตอน ตั้งแต่ Agent เริ่มเดิน, รับรู้สิ่งต่างๆ, ตัดสินใจ, และสุดท้ายเมื่อเดินไปถึงตำแหน่งที่มีสมบัติ มันก็จะเก็บสมบัติและการจำลองก็จะสิ้นสุดลง

**แผนภาพโครงสร้างคลาส (Class Diagram)ไดอะแกรมนี้แสดงให้เห็นว่า:Agent และ Treasure เป็น Thing ชนิดหนึ่ง (สืบทอดคุณสมบัติ)LineWorld เป็น Environment ชนิดหนึ่ง (สืบทอดคุณสมบัติ)LineWorld "มี" (contains) Thing หลายชิ้นอยู่ภายใน (รวมถึง Agent และ Treasure)Agent "ใช้" (uses) ฟังก์ชัน collector_program เพื่อตัดสินใจclassDiagram**
    class Thing {
        +location
    }
    class Agent {
        +program()
        +performance
    }
    class Treasure {
        
    }
    class Environment {
        +things[]
        +agents[]
        +add_thing()
        +run()
        +percept()*
        +execute_action()*
    }
    class LineWorld {
        +size
        +percept()
        +execute_action()
    }
    class "collector_program()" as Program {
        <<Function>>
        + (percepts) -> action
    }

    Thing <|-- Agent
    Thing <|-- Treasure
    Environment <|-- LineWorld

    LineWorld "1" -- "0..*" Thing : contains
    Agent "1" ..> "1" Program : uses
    
**2. แผนภาพลำดับการทำงาน (Sequence Diagram)ไดอะแกรมนี้แสดงวงจรการทำงาน (Simulation Loop) ใน 1 ขั้นตอน เมื่อ Agent ยังไม่เจอสมบัติ:run() ใน LineWorld เริ่มต้นขั้นตอนLineWorld เรียก percept() เพื่อดูว่า Agent รับรู้อะไรได้บ้าง (ในที่นี้คือไม่เจออะไร)LineWorld ส่งข้อมูลการรับรู้ (percepts) ไปให้ Agent เพื่อให้ program ทำงานcollector_program ประมวลผลและตัดสินใจส่งคืน action เป็น 'move_right'LineWorld รับ action มาแล้วเรียก execute_action() เพื่อปรับเปลี่ยนสถานะของโลก (เปลี่ยนตำแหน่งของ Agent)sequenceDiagram**
    participant User
    participant LW as LineWorld
    participant A as Agent
    participant P as collector_program

    User->>LW: run()
    loop แต่ละขั้นตอน (Step)
        LW->>LW: 1. percept(Agent)
        activate LW
        Note right of LW: Agent อยู่ที่ตำแหน่ง X <br/> ไม่เจอ Treasure <br/> percepts = []
        deactivate LW

        LW->>A: 2. program(percepts)
        activate A
        A->>P: 3. collector_program([])
        activate P
        P-->>A: 4. คืน Action: "move_right"
        deactivate P
        A-->>LW: คืน Action: "move_right"
        deactivate A

        LW->>LW: 5. execute_action("move_right")
        activate LW
        Note right of LW: เปลี่ยน Agent.location จาก X เป็น X+1
        deactivate LW
    end


In [30]:
import random

# --- ส่วนที่ 1: นิยามคลาสพื้นฐาน (Simplified versions from agents.py) ---
# ปกติเราจะ import มาจากไฟล์ agents.py แต่เพื่อให้ตัวอย่างนี้สมบูรณ์ในตัวเอง
# เราจะนิยามคลาสหลักๆ ที่จำเป็นขึ้นมาใหม่แบบง่ายๆ ที่นี่

class Thing:
    """
    คลาสแม่บทสำหรับวัตถุทุกชนิดที่สามารถปรากฏใน Environment ได้
    """
    def __repr__(self):
        return f'<{self.__class__.__name__}>'

In [31]:
class Agent(Thing):
    """
    Agent คือ Thing ที่มีความสามารถพิเศษ: มี 'โปรแกรม' ที่ใช้ตัดสินใจว่าจะทำอะไร
    โปรแกรมนี้จะรับข้อมูลการรับรู้ (percept) และส่งคืนเป็นการกระทำ (action)
    """
    def __init__(self, program=None):
        self.alive = True
        self.performance = 0
        if program is None:
            # ถ้าไม่กำหนดโปรแกรมมา จะให้ผู้ใช้ป้อน action เอง
            def program(percept):
                return input(f'Percept={percept}; action? ')
        self.program = program

    def is_alive(self):
        """คืนค่า True ถ้าเอเจนต์ยังมีชีวิตอยู่"""
        return self.alive

In [32]:
class Environment:
    """
    คลาสนามธรรมสำหรับสภาพแวดล้อม (โลก) ที่ Agent อาศัยอยู่
    """
    def __init__(self):
        self.things = []
        self.agents = []

    def add_thing(self, thing, location=None):
        """เพิ่ม Thing เข้าไปใน Environment"""
        if not isinstance(thing, Thing):
            thing = Agent(thing)
        thing.location = location if location is not None else self.default_location(thing)
        self.things.append(thing)
        if isinstance(thing, Agent):
            self.agents.append(thing)

    def delete_thing(self, thing):
        """ลบ Thing ออกจาก Environment อย่างปลอดภัย"""
        if thing in self.things:
            self.things.remove(thing)
        if thing in self.agents:
            self.agents.remove(thing)

    def list_things_at(self, location, tclass=Thing):
        """คืนค่ารายการของ Thing ทั้งหมดที่อยู่ในตำแหน่งที่กำหนด"""
        return [thing for thing in self.things if thing.location == location and isinstance(thing, tclass)]

    def run(self, steps=10):
        """เริ่มการจำลอง (Simulation) ตามจำนวนขั้นตอนที่กำหนด"""
        print(f"เริ่มต้นการจำลองใน {self.__class__.__name__} เป็นเวลา {steps} ขั้นตอน")
        for step in range(steps):
            if self.is_done():
                print("การจำลองสิ้นสุดลงแล้ว")
                return
            print(f"\n--- ขั้นตอนที่ {step+1} ---")
            for agent in self.agents:
                if agent.alive:
                    percept = self.percept(agent)
                    action = agent.program(percept)
                    self.execute_action(agent, action)
        print("\nการจำลองครบจำนวนขั้นตอนที่กำหนด")


    # --- เมธอดที่คลาสลูกของ Environment ต้อง implement ---
    def percept(self, agent):
        """ส่งคืนข้อมูลที่ Agent รับรู้ได้ ณ ตำแหน่งปัจจุบัน"""
        raise NotImplementedError

    def execute_action(self, agent, action):
        """เปลี่ยนแปลงสถานะของ Environment ตาม Action ที่ Agent ทำ"""
        raise NotImplementedError

    def is_done(self):
        """คืนค่า True เมื่อการจำลองสิ้นสุดเงื่อนไข"""
        return not any(agent.alive for agent in self.agents)
    
    def default_location(self, thing):
        """กำหนดตำแหน่งเริ่มต้นหากไม่ได้ระบุ"""
        return 0

In [33]:
# 2.1 สร้าง Thing ของเรา: สมบัติ (Treasure)
class Treasure(Thing):
    """สมบัติเป็น Thing ง่ายๆ ที่ไม่มีพฤติกรรมอะไรเป็นพิเศษ"""
    pass

# 2.2 สร้าง Environment ของเรา: โลกเส้นตรง (LineWorld)
class LineWorld(Environment):
    """
    โลกของเราเป็นเส้นตรงที่มีขนาด 10 ช่อง (ตำแหน่ง 0 ถึง 9)
    """
    def __init__(self, size=10):
        super().__init__()
        self.size = size
        print(f"สร้าง LineWorld ขนาด {size} ช่อง (0-{size-1})")

    def percept(self, agent):
        """Agent จะรับรู้ถึง 'Thing' ทั้งหมดที่อยู่ในตำแหน่งเดียวกับมัน"""
        things_at_location = self.list_things_at(agent.location)
        print(f"Agent ที่ตำแหน่ง {agent.location} รับรู้ได้ว่ามี: {things_at_location}")
        return things_at_location

    def execute_action(self, agent, action):
        """จัดการกับการกระทำของ Agent"""
        print(f"Agent ตัดสินใจทำ Action: '{action}'")
        if action == "move_right":
            if agent.location < self.size - 1:
                agent.location += 1
                agent.performance -= 1 # เสียคะแนนในการเคลื่อนที่
                print(f"Agent เคลื่อนที่ไปทางขวา ตอนนี้อยู่ที่ตำแหน่ง {agent.location}")
            else:
                print("Agent อยู่ขวาสุดแล้ว เคลื่อนที่ต่อไม่ได้")
        elif action == "collect":
            treasures = self.list_things_at(agent.location, tclass=Treasure)
            if treasures:
                treasure = treasures[0]
                self.things.remove(treasure) # นำสมบัติออกจากโลก
                agent.performance += 100 # ได้คะแนนเมื่อเก็บสมบัติ
                print(f"Agent เก็บ {treasure} ได้สำเร็จ! คะแนนรวม: {agent.performance}")
            else:
                print("ไม่มีสมบัติให้เก็บที่ตำแหน่งนี้")

    def is_done(self):
        """การจำลองจะจบลงเมื่อไม่มีสมบัติเหลืออยู่ในโลกอีกต่อไป"""
        return not any(isinstance(t, Treasure) for t in self.things)

In [34]:
# --- ส่วนที่ 3: สร้างโปรแกรมสำหรับ Agent ---

def collector_program(percepts):
    """
    โปรแกรมอย่างง่ายสำหรับ CollectorAgent
    - ถ้าเจอ Treasure ใน percepts ให้ทำการ 'collect'
    - ถ้าไม่เจออะไร ให้ 'move_right'
    """
    # ตรวจสอบว่าในรายการ percepts มี Treasure อยู่หรือไม่
    if any(isinstance(p, Treasure) for p in percepts):
        return "collect"
    else:
        return "move_right"


In [35]:
# --- ส่วนที่ 4: เริ่มการจำลอง ---
if __name__ == "__main__":
    # 1. สร้าง Environment
    world = LineWorld(size=10)

    # 2. สร้าง Agent พร้อมกับ Program
    agent = Agent(program=collector_program)

    # 3. สร้าง Thing (สมบัติ)
    treasure = Treasure()

    # 4. เพิ่ม Agent และ Treasure เข้าไปในโลก ณ ตำแหน่งที่กำหนด
    #    Agent เริ่มที่ตำแหน่ง 0, สมบัติอยู่ที่ตำแหน่ง 7
    world.add_thing(agent, location=0)
    treasure2 = Treasure()
    world.add_thing(treasure, location=3)
    world.add_thing(treasure2, location=7)
    print(f"เพิ่ม {agent} ที่ตำแหน่ง {agent.location} และ {treasure} ที่ตำแหน่ง {treasure.location} และ {treasure2} ที่ตำแหน่ง {treasure2.location}")

    # 5. เริ่มการจำลอง
    world.run(steps=15)

    # 6. แสดงผลลัพธ์สุดท้าย
    print("\n--- ผลลัพธ์สุดท้าย ---")
    print(f"คะแนนสุดท้ายของ Agent: {agent.performance}")
    print(f"สิ่งของที่เหลืออยู่ในโลก: {world.things}")

สร้าง LineWorld ขนาด 10 ช่อง (0-9)
เพิ่ม <Agent> ที่ตำแหน่ง 0 และ <Treasure> ที่ตำแหน่ง 3 และ <Treasure> ที่ตำแหน่ง 7
เริ่มต้นการจำลองใน LineWorld เป็นเวลา 15 ขั้นตอน

--- ขั้นตอนที่ 1 ---
Agent ที่ตำแหน่ง 0 รับรู้ได้ว่ามี: [<Agent>]
Agent ตัดสินใจทำ Action: 'move_right'
Agent เคลื่อนที่ไปทางขวา ตอนนี้อยู่ที่ตำแหน่ง 1

--- ขั้นตอนที่ 2 ---
Agent ที่ตำแหน่ง 1 รับรู้ได้ว่ามี: [<Agent>]
Agent ตัดสินใจทำ Action: 'move_right'
Agent เคลื่อนที่ไปทางขวา ตอนนี้อยู่ที่ตำแหน่ง 2

--- ขั้นตอนที่ 3 ---
Agent ที่ตำแหน่ง 2 รับรู้ได้ว่ามี: [<Agent>]
Agent ตัดสินใจทำ Action: 'move_right'
Agent เคลื่อนที่ไปทางขวา ตอนนี้อยู่ที่ตำแหน่ง 3

--- ขั้นตอนที่ 4 ---
Agent ที่ตำแหน่ง 3 รับรู้ได้ว่ามี: [<Agent>, <Treasure>]
Agent ตัดสินใจทำ Action: 'collect'
Agent เก็บ <Treasure> ได้สำเร็จ! คะแนนรวม: 97

--- ขั้นตอนที่ 5 ---
Agent ที่ตำแหน่ง 3 รับรู้ได้ว่ามี: [<Agent>]
Agent ตัดสินใจทำ Action: 'move_right'
Agent เคลื่อนที่ไปทางขวา ตอนนี้อยู่ที่ตำแหน่ง 4

--- ขั้นตอนที่ 6 ---
Agent ที่ตำแหน่ง 4 รับรู้ได้ว่ามี: [<Ag

## SIMPLE AGENT AND ENVIRONMENT

Let's begin by using the `Agent` class to creating our first agent - a blind dog.

In [36]:
class BlindDog(Agent):
    def eat(self, thing):
        print("Dog: Ate food at {}.".format(self.location))
            
    def drink(self, thing):
        print("Dog: Drank water at {}.".format( self.location))

dog = BlindDog()

What we have just done is create a dog who can only feel what's in his location (since he's blind), and can eat or drink. Let's see if he's alive...

In [37]:
print(dog.alive)

True


![Cool dog](https://gifgun.files.wordpress.com/2015/07/wpid-wp-1435860392895.gif)
This is our dog. How cool is he? Well, he's hungry and needs to go search for food. For him to do this, we need to give him a program. But before that, let's create a park for our dog to play in.

### ENVIRONMENT - Park

A park is an example of an environment because our dog can perceive and act upon it. The <b>Environment</b> class is an abstract class, so we will have to create our own subclass from it before we can use it.

In [38]:
class Food(Thing):
    pass

class Water(Thing):
    pass

class Park(Environment):
    def percept(self, agent):
        '''return a list of things that are in our agent's location'''
        things = self.list_things_at(agent.location)
        return things
    
    def execute_action(self, agent, action):
        '''changes the state of the environment based on what the agent does.'''
        if action == "move down":
            print("{} moved down - agents.ipynb:16".format(str(agent)[1:-1]))
            agent.movedown()
        elif action == "eat":
            items = self.list_things_at(agent.location, tclass=Food)
            if len(items) != 0:
                if agent.eat(items[0]): #Have the dog eat the first item
                    print('{} ate {} at location: {} - agents.ipynb:22'
                          .format(str(agent)[1:-1], str(items[0])[1:-1], agent.location))
                    self.delete_thing(items[0]) #Delete it from the Park after.
        elif action == "drink":
            items = self.list_things_at(agent.location, tclass=Water)
            if len(items) != 0:
                if agent.drink(items[0]): #Have the dog drink the first item
                    print('{} drank {} at location: {} - agents.ipynb:29'
                          .format(str(agent)[1:-1], str(items[0])[1:-1], agent.location))
                    self.delete_thing(items[0]) #Delete it from the Park after.

    def is_done(self):
        '''By default, we're done when we can't find a live agent, 
        but to prevent killing our cute dog, we will stop before itself - when there is no more food or water'''
        no_edibles = not any(isinstance(thing, Food) or isinstance(thing, Water) for thing in self.things)
        dead_agents = not any(agent.is_alive() for agent in self.agents)
        return dead_agents or no_edibles


### PROGRAM - BlindDog
Now that we have a <b>Park</b> Class, we re-implement our <b>BlindDog</b> to be able to move down and eat food or drink water only if it is present.


In [39]:
class BlindDog(Agent):
    location = 1
    
    def movedown(self):
        self.location += 1

    def moveup(self):
        self.location[1] -= 1

    def moveleft(self):
        self.location[0] -= 1

    def moveright(self):
        self.location[0] += 1
        
    def eat(self, thing):
        '''returns True upon success or False otherwise'''
        if isinstance(thing, Food):
            return True
        return False
    
    def drink(self, thing):
        ''' returns True upon success or False otherwise'''
        if isinstance(thing, Water):
            return True
        return False

Now its time to implement a <b>program</b> module for our dog. A program controls how the dog acts upon its environment. Our program will be very simple, and is shown in the table below.
<table>
    <tr>
        <td><b>Percept:</b> </td>
        <td>Feel Food </td>
        <td>Feel Water</td>
        <td>Feel Nothing</td>
   </tr>
   <tr>
       <td><b>Action:</b> </td>
       <td>eat</td>
       <td>drink</td>
       <td>move down</td>
   </tr>
        
</table>

In [40]:
def program(percepts):
    '''Returns an action based on the dog's percepts'''
    for p in percepts:
        if isinstance(p, Food):
            return 'eat'
        elif isinstance(p, Water):
            return 'drink'
    return 'move down'

Let's now run our simulation by creating a park with some food, water, and our dog.

In [41]:
park = Park() # environment for our dog to live in
dog = BlindDog(program) 
dogfood = Food()
water = Water()
park.add_thing(dog, 1)
park.add_thing(dogfood, 5)
park.add_thing(water, 7)

park.run(5)

เริ่มต้นการจำลองใน Park เป็นเวลา 5 ขั้นตอน

--- ขั้นตอนที่ 1 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 2 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 3 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 4 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 5 ---
BlindDog ate Food at location: 5 - agents.ipynb:22

การจำลองครบจำนวนขั้นตอนที่กำหนด


Notice that the dog moved from location 1 to 4, over 4 steps, and ate food at location 5 in the 5th step.

Let's continue this simulation for 5 more steps.

In [42]:
park.run(5)

เริ่มต้นการจำลองใน Park เป็นเวลา 5 ขั้นตอน

--- ขั้นตอนที่ 1 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 2 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 3 ---
BlindDog drank Water at location: 7 - agents.ipynb:29
การจำลองสิ้นสุดลงแล้ว


Perfect! Note how the simulation stopped after the dog drank the water - exhausting all the food and water ends our simulation, as we had defined before. Let's add some more water and see if our dog can reach it.

In [43]:
park.add_thing(water, 15)
park.run(10)

เริ่มต้นการจำลองใน Park เป็นเวลา 10 ขั้นตอน

--- ขั้นตอนที่ 1 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 2 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 3 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 4 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 5 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 6 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 7 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 8 ---
BlindDog moved down - agents.ipynb:16

--- ขั้นตอนที่ 9 ---
BlindDog drank Water at location: 15 - agents.ipynb:29
การจำลองสิ้นสุดลงแล้ว


Above, we learnt to implement an agent, its program, and an environment on which it acts. However, this was a very simple case. Let's try to add complexity to it by creating a 2-Dimensional environment!


## AGENTS IN A 2D ENVIRONMENT

For us to not read so many logs of what our dog did, we add a bit of graphics while making our Park 2D. To do so, we will need to make it a subclass of <b>GraphicEnvironment</b> instead of Environment. Parks implemented by subclassing <b>GraphicEnvironment</b> class adds these extra properties to it:

 - Our park is indexed in the 4th quadrant of the X-Y plane.
 - Every time we create a park subclassing <b>GraphicEnvironment</b>, we need to define the colors of all the things we plan to put into the park. The colors are defined in typical [<b>RGB digital 8-bit format</b>](https://en.wikipedia.org/wiki/RGB_color_model#Numeric_representations), common across the web.
 - Fences are added automatically to all parks so that our dog does not go outside the park's boundary - it just isn't safe for blind dogs to be outside the park by themselves! <b>GraphicEnvironment</b> provides `is_inbounds` function to check if our dog tries to leave the park.
 
First let us try to upgrade our 1-dimensional `Park` environment by just replacing its superclass by `GraphicEnvironment`. 

In [44]:
class Park2D(GraphicEnvironment):

    def percept(self, agent):
        '''return a list of things that are in our agent's location'''
        things = self.list_things_at(agent.location)
        return things

    def execute_action(self, agent, action):
        '''changes the state of the environment based on what the agent does.'''

        if action == "turnright":
            print("{} turned right - agents.ipynb:12".format(str(agent)[1:-1]))
            agent.turn(Direction.R)

        elif action == "turnleft":
            print("{} turned left - agents.ipynb:16".format(str(agent)[1:-1]))
            agent.turn(Direction.L)

        elif action == "moveforward":
            print("{} moved forward - agents.ipynb:20".format(str(agent)[1:-1]))
            agent.moveforward()

        elif action == "move down":
            print("{} moved down - agents.ipynb:24".format(str(agent)[1:-1]))
            agent.movedown()

        elif action == "eat":
            items = self.list_things_at(agent.location, tclass=Food)

            if len(items) != 0:
                if agent.eat(items[0]):
                    print("{} ate Food - agents.ipynb:32".format(str(agent)[1:-1]))
                    self.delete_thing(items[0])

        elif action == "drink":
            items = self.list_things_at(agent.location, tclass=Water)

            if len(items) != 0:
                if agent.drink(items[0]):
                    print("{} drank Water - agents.ipynb:40".format(str(agent)[1:-1]))
                    self.delete_thing(items[0])

        elif action == "grab":
            items = self.list_things_at(agent.location, tclass=Gold)

            if len(items) != 0:
                print("{} grabbed Gold - agents.ipynb:47".format(str(agent)[1:-1]))
                self.delete_thing(items[0])
                agent.performance += 100

    def is_done(self):
        no_edibles = not any(isinstance(t, Food) or isinstance(t, Water)
                             for t in self.things)

        dead_agents = not any(agent.is_alive()
                              for agent in self.agents)

        return dead_agents or no_edibles

In [45]:
class BlindDog(Agent):

    location = [0, 1]
    direction = Direction("down")

    def movedown(self):
        self.location[1] += 1

    def moveforward(self):
        if self.direction.direction == Direction.R:
            self.location[0] += 1
        elif self.direction.direction == Direction.L:
            self.location[0] -= 1
        elif self.direction.direction == Direction.U:
            self.location[1] -= 1
        elif self.direction.direction == Direction.D:
            self.location[1] += 1

    def eat(self, thing):
        return isinstance(thing, Food)

    def drink(self, thing):
        return isinstance(thing, Water)

In [46]:
from random import choice
import random
def program(percepts):
    '''Returns an action based on its percepts'''
    # ตรวจสอบเพื่อกินหรือดื่มก่อน
    for p in percepts:
        if isinstance(p, Food):
            return 'eat'
        elif isinstance(p, Water):
            return 'drink'

    # หากไม่เจออาหารหรือน้ำ ให้ตรวจสอบการชน
    # (Bump จะต้องเป็นหนึ่งในสิ่งที่ Agent รับรู้ได้)
    if any(isinstance(p, Bump) for p in percepts):
            return "turnright"
    else:
        # ถ้าไม่ชน สามารถเดินหน้า, เลี้ยวซ้าย, หรือเลี้ยวขวาได้
        choice = random.choice((1, 2, 3, 4))

    # ตัดสินใจเลือกการกระทำ
    if choice == 1:
        return 'turnright'
    elif choice == 2:
        return 'turnleft'
    else:
        return 'moveforward'

Now let's test this new park with our same dog, food and water. We color our dog with a nice red and mark food and water with orange and blue respectively.

In [47]:
park = Park2D(5,20, color={'BlindDog': (200,0,0), 'Agent': (200,0,0), 'Water': (0, 200, 200), 'Food': (230, 115, 40)}) # park width is set to 5, and height to 20
dog = BlindDog(program)
dogfood = Food()
water = Water()
park.add_thing(dog, [0,1])
park.add_thing(dogfood, [0,5])
park.add_thing(water, [0,7])
morewater = Water()
park.add_thing(morewater, [0,15])
print("BlindDog starts at (1,1) facing downwards, lets see if he can find any food!  agents2.ipynb:10 - agents.ipynb:10")
park.run(20)
#การเพิ่มสีสำหรับคลาส Agent เข้าไปใน dictionary ตอนที่สร้าง Park2D

,,,,
,,,,
,,,,
,,,,
,,,,
,,,,
,,,,
,,,,
,,,,
,,,,
,,,,


Adding some graphics was a good idea! We immediately see that the code works, but our blind dog doesn't make any use of the 2 dimensional space available to him. Let's make our dog more energetic so that he turns and moves forward, instead of always moving down. In doing so, we'll also need to make some changes to our environment to be able to handle this extra motion.

### PROGRAM - EnergeticBlindDog

Let's make our dog turn or move forwards at random - except when he's at the edge of our park - in which case we make him change his direction explicitly by turning to avoid trying to leave the park. However, our dog is blind so he wouldn't know which way to turn - he'd just have to try arbitrarily.

<table>
    <tr>
        <td><b>Percept:</b> </td>
        <td>Feel Food </td>
        <td>Feel Water</td>
        <td>Feel Nothing</td>
   </tr>
   <tr>
       <td><b>Action:</b> </td>
       <td>eat</td>
       <td>drink</td>
       <td>
       <table>
           <tr>
               <td><b>Remember being at Edge : </b></td>
               <td>At Edge</td>
               <td>Not at Edge</td>
           </tr>
           <tr>
               <td><b>Action : </b></td>
               <td>Turn Left / Turn Right <br> ( 50% - 50% chance )</td>
               <td>Turn Left / Turn Right / Move Forward <br> ( 25% - 25% - 50% chance )</td>
           </tr>
       </table>
       </td>
   </tr>
        
</table>

In [48]:
import random
from agents import Agent as AimaAgent, Thing as AimaThing, Direction, Bump

# Redefine these for the 2D section so they are compatible with GraphicEnvironment
class Food(AimaThing):
    pass

class Water(AimaThing):
    pass

class EnergeticBlindDog(AimaAgent):
    location = [0, 1]
    direction = Direction("down")
    
    def moveforward(self, success=True):
        """moveforward possible only if success (i.e. valid destination location)"""
        if not success:
            return
        if self.direction.direction == Direction.R:
            self.location[0] += 1
        elif self.direction.direction == Direction.L:
            self.location[0] -= 1
        elif self.direction.direction == Direction.D:
            self.location[1] += 1
        elif self.direction.direction == Direction.U:
            self.location[1] -= 1
    
    def turn(self, d):
        self.direction = self.direction + d
        
    def eat(self, thing):
        """returns True upon success or False otherwise"""
        return isinstance(thing, Food)
    
    def drink(self, thing):
        """returns True upon success or False otherwise"""
        return isinstance(thing, Water)

def program(percepts):
    """Returns an action based on its percepts"""
    # Default behavior when nothing special is perceived
    choice = random.choice((1, 2, 3, 4))  # 1-right, 2-left, others-forward
    
    for p in percepts:
        if isinstance(p, Food):
            return 'eat'
        elif isinstance(p, Water):
            return 'drink'
        elif isinstance(p, Bump):
            choice = random.choice((1, 2))  # turn only at boundary

    if choice == 1:
        return 'turnright'
    elif choice == 2:
        return 'turnleft'
    else:
        return 'moveforward'

### ENVIRONMENT - Park2D

We also need to modify our park accordingly, in order to be able to handle all the new actions our dog wishes to execute. Additionally, we'll need to prevent our dog from moving to locations beyond our park boundary - it just isn't safe for blind dogs to be outside the park by themselves.

In [49]:
class Park2D(GraphicEnvironment):
    def percept(self, agent):
        '''return a list of things that are in our agent's location'''
        things = self.list_things_at(agent.location)
        loc = list(agent.location)  # target location copy
        # Check if agent is about to bump into a wall
        if agent.direction.direction == Direction.R:
            loc[0] += 1
        elif agent.direction.direction == Direction.L:
            loc[0] -= 1
        elif agent.direction.direction == Direction.D:
            loc[1] += 1
        elif agent.direction.direction == Direction.U:
            loc[1] -= 1
        if not self.is_inbounds(loc):
            things.append(Bump())
        return things
    
    def execute_action(self, agent, action):
        '''changes the state of the environment based on what the agent does.'''
        if action == 'turnright':
            print('{} decided to {} at location: {} - agents.ipynb:22'.format(str(agent)[1:-1], action, agent.location))
            agent.turn(Direction.R)
        elif action == 'turnleft':
            print('{} decided to {} at location: {} - agents.ipynb:25'.format(str(agent)[1:-1], action, agent.location))
            agent.turn(Direction.L)
        elif action == 'moveforward':
            print('{} decided to move {}wards at location: {} - agents.ipynb:28'.format(str(agent)[1:-1], agent.direction.direction, agent.location))
            agent.moveforward()
        elif action == "eat":
            items = self.list_things_at(agent.location, tclass=Food)
            if len(items) != 0 and agent.eat(items[0]):
                print('{} ate {} at location: {} - agents.ipynb:33'
                      .format(str(agent)[1:-1], str(items[0])[1:-1], agent.location))
                self.delete_thing(items[0])
        elif action == "drink":
            items = self.list_things_at(agent.location, tclass=Water)
            if len(items) != 0 and agent.drink(items[0]):
                print('{} drank {} at location: {} - agents.ipynb:39'
                      .format(str(agent)[1:-1], str(items[0])[1:-1], agent.location))
                self.delete_thing(items[0])
                    
    def is_done(self):
        '''By default, we're done when we can't find a live agent, 
        but to prevent killing our cute dog, we will stop before itself - when there is no more food or water'''
        no_edibles = not any(isinstance(thing, Food) or isinstance(thing, Water) for thing in self.things)
        dead_agents = not any(agent.is_alive() for agent in self.agents)
        return dead_agents or no_edibles

In [50]:
import random
class SmartBlindDog(Agent):

    def __init__(self, program):
        super().__init__(program)
        self.visited = set()
        self.step = 0 
        self.change_direction = False
        self.scan_dir = "down"

In [51]:
import random

def smart_program(percepts):

    # เจออาหาร
    if any(isinstance(p, Food) for p in percepts):
        dog.change_direction = True
        return "eat"

    # เจอน้ำ
    if any(isinstance(p, Water) for p in percepts):
        return "drink"

    # จำตำแหน่งปัจจุบัน
    dog.visited.add(tuple(dog.location))

    # ชนกำแพง
    if any(isinstance(p, Bump) for p in percepts):
        return random.choice(["turnleft", "turnright"])

    # คำนวณตำแหน่งข้างหน้า
    x, y = dog.location

    if dog.direction.direction == Direction.R:
        dog.change_direction = False
        return random.choice(["turnleft", "turnright"])
        next_loc = (x + 1, y)
    elif dog.direction.direction == Direction.L:
        next_loc = (x - 1, y)
    elif dog.direction.direction == Direction.U:
        next_loc = (x, y - 1)
    else:  # Direction.D
        next_loc = (x, y + 1)

    # ถ้ายังไม่เคยไป
    if next_loc not in dog.visited:
        return "moveforward"

    # เคยไปแล้ว -> ส่วนใหญ่จะเปลี่ยนทิศ
    if random.random() < 0.8:
        return random.choice(["turnleft", "turnright"])

    # มีโอกาส 20% เดินต่อ (เผื่อต้องย้อนกลับ)
    return "moveforward"

Now that our park is ready for the 2D motion of our energetic dog, lets test it!

In [52]:
park = Park2D(5,5, color={'EnergeticBlindDog': (200,0,0), 'Water': (0, 200, 200), 'Food': (230, 115, 40)})
dog = EnergeticBlindDog(program)
dogfood = Food()
water = Water()
park.add_thing(dog, [0,0])
park.add_thing(dogfood, [1,2])
park.add_thing(water, [0,1])
morewater = Water()
morefood = Food()
park.add_thing(morewater, [2,4])
park.add_thing(morefood, [4,3])
print("dog started at [0,0], facing down. Let's see if he found any food or water! - agents.ipynb:12")
park.run(10)

,,,,
,,,,
,,,,
,,,,
,,,,


HW2









## Wumpus Environment

In [101]:
from ipythonblocks import BlockGrid
from agents import *
from IPython.display import clear_output
import time
import random

color = {"Breeze": (225, 225, 225),
        "Pit": (0,0,0),
        "Gold": (253, 208, 23),
        "Glitter": (253, 208, 23),
        "Wumpus": (43, 27, 23),
        "Stench": (128, 128, 128),
        "Explorer": (0, 0, 255),
        "Wall": (44, 53, 57)
        }

# ---------- ระบบทิศทาง: 0=North, 1=East, 2=South, 3=West ----------
DIR_VECTORS = {0: (0, 1), 1: (1, 0), 2: (0, -1), 3: (-1, 0)}

def turns_needed(current_dir, target_dir):
    """คำนวณคำสั่งเลี้ยวที่ต้องใช้ เพื่อหันจาก current_dir ไปเป็น target_dir"""
    diff = (target_dir - current_dir) % 4
    if diff == 0:
        return []
    elif diff == 1:
        return ['TurnRight']
    elif diff == 2:
        return ['TurnRight', 'TurnRight']
    else:
        return ['TurnLeft']

# ---------- ตัวแปรเก็บสถานะของ agent ----------
agent_state = {
    "has_gold": False,
    "climbed": False,
    "action_queue": [],
    "pos": (0, 0),
    "direction": 0,
    "path_history": [],      # ทิศที่เดินหน้าสำเร็จมาแล้ว (ใช้ย้อนกลับ)
    "tried_dirs": {},        # pos -> set ทิศที่เคยลองเดินแล้วชนกำแพง
    "dead_ends": set(),      # ตำแหน่งที่จำไว้ว่า "ตัน" ห้ามเดินเข้าไปอีก
    "pending_forward": None  # (pos, direction) ของการเดินหน้าที่รอผลตรวจสอบ
}

def do_action(action):
    """อัปเดตสถานะภายใน (ตำแหน่ง/ทิศทาง) ตามคำสั่งที่กำลังจะส่งออกไป"""
    if action == 'TurnRight':
        agent_state["direction"] = (agent_state["direction"] + 1) % 4
    elif action == 'TurnLeft':
        agent_state["direction"] = (agent_state["direction"] - 1) % 4
    elif action == 'Forward':
        agent_state["pending_forward"] = (agent_state["pos"], agent_state["direction"])
    return action

def safe_wumpus_program(percepts):
    print(f"สิ่งที่รับรู้ตอนนี้: {percepts} - agents.ipynb:56")

    # 0. ตรวจผลของการเดินหน้าครั้งก่อน (สำเร็จ หรือ ชนกำแพง)
    if agent_state["pending_forward"] is not None:
        prev_pos, prev_dir = agent_state["pending_forward"]
        agent_state["pending_forward"] = None
        if 'Bump' in percepts:
            tried = agent_state["tried_dirs"].setdefault(prev_pos, set())
            tried.add(prev_dir)
        else:
            dx, dy = DIR_VECTORS[prev_dir]
            agent_state["pos"] = (prev_pos[0] + dx, prev_pos[1] + dy)
            agent_state["path_history"].append(prev_dir)

    # ถ้ามีคำสั่งค้างอยู่ในคิว ให้ทำต่อจนกว่าจะหมดคิว
    if agent_state["action_queue"]:
        return do_action(agent_state["action_queue"].pop(0))

    # 1. เจอประกายทองคำ -> หยิบทันที
    if 'Glitter' in percepts:
        agent_state["has_gold"] = True
        return do_action('Grab')

    # 2. มีทองแล้ว -> เดินย้อนเส้นทางเดิมกลับไปจุดเริ่มต้น
    if agent_state["has_gold"]:
        if not agent_state["path_history"]:
            agent_state["climbed"] = True
            return do_action('Climb')
        d = agent_state["path_history"].pop()
        opp = (d + 2) % 4
        seq = turns_needed(agent_state["direction"], opp) + ['Forward']
        agent_state["action_queue"] = seq
        return do_action(agent_state["action_queue"].pop(0))

    pos = agent_state["pos"]
    direction = agent_state["direction"]
    tried = agent_state["tried_dirs"].get(pos, set())

    # 3. เจอกำแพง -> เช็คว่า "ตัน" หรือยัง
    #    ลองมาแล้ว 3 ทิศ (จาก 4) = ตัน เพราะทิศที่เดินเข้ามาการันตีว่าเปิดอยู่แล้ว
    if 'Bump' in percepts:
        if len(tried) >= 3 and agent_state["path_history"]:
            agent_state["dead_ends"].add(pos)     # จำตำแหน่งนี้ว่าตัน
            d = agent_state["path_history"].pop()
            opp = (d + 2) % 4
            seq = turns_needed(direction, opp) + ['Forward']  # ถอยกลับช่องก่อนหน้า
            agent_state["action_queue"] = seq
            return do_action(agent_state["action_queue"].pop(0))
        else:
            turn = random.choice(['TurnRight', 'TurnLeft'])
            agent_state["action_queue"] = [turn, 'Forward']
            return do_action(agent_state["action_queue"].pop(0))

    # 4. ช่องข้างหน้าเคยรู้ว่าตัน -> เลี้ยวหนีแทนที่จะเดินเข้าไปซ้ำ
    dx, dy = DIR_VECTORS[direction]
    next_pos = (pos[0] + dx, pos[1] + dy)
    if next_pos in agent_state["dead_ends"]:
        return do_action(random.choice(['TurnRight', 'TurnLeft']))

    # 5. ทางโล่ง ไม่เคยรู้ว่าตัน -> เดินหน้าสำรวจต่อ
    return do_action('Forward')


# สร้างโลก Wumpus
w = WumpusEnvironment(safe_wumpus_program, 7, 7)
grid = BlockGrid(w.width, w.height, fill=(123, 234, 123))

def draw_grid(world):
    global grid
    grid[:] = (123, 234, 123)
    for x in range(0, len(world)):
        for y in range(0, len(world[x])):
            if len(world[x][y]):
                grid[y, x] = color[world[x][y][-1].__class__.__name__]

def step(total_steps=100, delay=0.3):
    global grid, w
    for i in range(total_steps):
        draw_grid(w.get_world())
        clear_output(wait=True)
        grid.show()
        w.step()
        time.sleep(delay)
        if agent_state["climbed"]:
            break

In [102]:
step()

,,,,,,
,,,,,,
,,,,,,
,,,,,,
,,,,,,
,,,,,,
,,,,,,


สิ่งที่รับรู้ตอนนี้: [[None], [<Bump>, <Stench>], [<Bump>], [None], [<Stench>, <Glitter>, None]] - agents.ipynb:56


KeyboardInterrupt: 